
# Assignment 1 — Build a Custom Missing-Value Imputer

**Course:** Feature Engineering & MLOps  
**Assignment:** 1  
**Topic:** Custom Imputer Class (Missing Value Handling)

## Objective

Design and implement a custom Python imputer that follows scikit-learn's `fit()` / `transform()` pattern, apply it to the PrepEdge student-performance dataset, verify the results, and answer the required reflection questions.


## 1. Imports and Dataset Setup

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# The assignment repository path is preferred.
# The /mnt/data fallback is used only so this uploaded-data notebook can be executed here.
DATA_PATH = Path("data/raw/student_performance_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/student_performance_raw.csv")

assert DATA_PATH.exists(), f"Dataset not found. Expected: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Shape: {df.shape}")
print("\nData types:")
display(df.dtypes.to_frame("dtype"))


Dataset path: /mnt/data/student_performance_raw.csv
Shape: (600, 17)

Data types:


,dtype
student_id,int64
city,object
city_tier,int64
course,object
batch_type,object
age,int64
enrollment_date,object
attendance_pct,float64
weekly_study_hours,float64
income_bracket,object


## 2. Initial Missing-Value Analysis

In [2]:

missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(
          missing_pct=lambda x: 100 * x["missing_count"] / len(df)
      )
)

display(missing_summary[missing_summary["missing_count"] > 0])

print("Expected missing-value columns from the assignment:")
print("- weekly_study_hours → MCAR, numeric")
print("- prev_exam_score → MAR, numeric")
print("- mock_test_3 → MNAR, numeric")
print("- income_bracket → missing categorical values")
print("- feedback_text → missing free-text values")


,missing_count,missing_pct
weekly_study_hours,36,6.0000
income_bracket,30,5.0000
prev_exam_score,25,4.1667
mock_test_3,21,3.5000
feedback_text,68,11.3333


Expected missing-value columns from the assignment:
- weekly_study_hours → MCAR, numeric
- prev_exam_score → MAR, numeric
- mock_test_3 → MNAR, numeric
- income_bracket → missing categorical values
- feedback_text → missing free-text values



## 3. Train/Test Split



In [3]:

X_train, X_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print(f"Training shape: {X_train.shape}")
print(f"Test shape:     {X_test.shape}")
print(f"Training proportion: {len(X_train) / len(df):.0%}")
print(f"Test proportion:     {len(X_test) / len(df):.0%}")

print("\nMissing values in training data:")
display(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nMissing values in test data:")
display(X_test.isna().sum()[X_test.isna().sum() > 0])


Training shape: (480, 17)
Test shape:     (120, 17)
Training proportion: 80%
Test proportion:     20%

Missing values in training data:


weekly_study_hours    25
income_bracket        24
prev_exam_score       23
mock_test_3           15
feedback_text         54
dtype: int64


Missing values in test data:


weekly_study_hours    11
income_bracket         6
prev_exam_score        2
mock_test_3            6
feedback_text         14
dtype: int64


## 4. Implement `CustomImputer`

### Design decisions

- Numeric columns are detected with `pandas.api.types.is_numeric_dtype`.
- Non-numeric columns (including the dataset's categorical and free-text columns) use the categorical strategy.
- Numeric fill values are calculated with the requested mean/median strategy.
- Non-numeric fill values use the most frequent value.
- Statistics are learned once in `fit()` and stored in `fill_values_`.
- Missing indicators are based on the columns that had missing values in the **training data**.
- `transform()` never recomputes fill statistics.
- `check_is_fitted()` protects against calling `transform()` before `fit()`.
- The implementation validates that the columns seen at transform time match the columns seen during fit, which makes schema drift explicit rather than silently producing an incorrect result.


In [ ]:

class CustomImputer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="most_frequent",
        add_missing_indicator=True,
        column_overrides=None,
    ):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _validate_strategies(self):
        """Validate the configured dataset-wide and per-column strategies."""
        if self.numeric_strategy not in {"mean", "median"}:
            raise ValueError("numeric_strategy must be 'mean' or 'median'.")
        if self.categorical_strategy != "most_frequent":
            raise ValueError("categorical_strategy must be 'most_frequent'.")
        if self.column_overrides is not None and not isinstance(self.column_overrides, dict):
            raise TypeError("column_overrides must be a dictionary or None.")

        for column, strategy in (self.column_overrides or {}).items():
            if strategy not in {"mean", "median", "most_frequent"}:
                raise ValueError(
                    f"Invalid override for '{column}': {strategy!r}. "
                    "Use 'mean', 'median', or 'most_frequent'."
                )

    def _effective_strategy(self, column):
        """Return the strategy that applies to a particular column."""
        return (self.column_overrides or {}).get(
            column,
            self.numeric_strategy
            if pd.api.types.is_numeric_dtype(self._fit_dtypes[column])
            else self.categorical_strategy,
        )

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CustomImputer.fit expects a pandas DataFrame.")

        self._validate_strategies()

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self._fit_dtypes = X.dtypes.to_dict()
        self.missing_indicator_columns_ = [
            col for col in X.columns if X[col].isna().any()
        ]

        fill_values = {}
        strategies = {}

        for column in X.columns:
            strategy = self._effective_strategy(column)
            strategies[column] = strategy

            if pd.api.types.is_numeric_dtype(X[column]):
                if strategy == "mean":
                    fill_value = X[column].mean()
                elif strategy == "median":
                    fill_value = X[column].median()
                elif strategy == "most_frequent":
                    fill_value = X[column].mode(dropna=True).iloc[0]
                else:
                    raise ValueError(
                        f"Strategy {strategy!r} is not valid for numeric column '{column}'."
                    )
            else:
                if strategy != "most_frequent":
                    raise ValueError(
                        f"Non-numeric column '{column}' must use 'most_frequent'."
                    )
                modes = X[column].mode(dropna=True)
                fill_value = modes.iloc[0] if not modes.empty else np.nan

            fill_values[column] = fill_value

        self.fill_values_ = fill_values
        self.strategies_ = strategies
        return self

    def transform(self, X):
        check_is_fitted(
            self,
            attributes=["fill_values_", "feature_names_in_", "missing_indicator_columns_"]
        )

        if not isinstance(X, pd.DataFrame):
            raise TypeError("CustomImputer.transform expects a pandas DataFrame.")

        expected = list(self.feature_names_in_)
        actual = list(X.columns)

        if actual != expected:
            missing = [col for col in expected if col not in actual]
            extra = [col for col in actual if col not in expected]
            raise ValueError(
                "Input columns do not match the columns seen during fit. "
                f"Missing columns: {missing}; Extra columns: {extra}."
            )

        X_out = X.copy()

        # Capture missingness BEFORE filling so the indicators represent the
        # original missingness in the data being transformed.
        if self.add_missing_indicator:
            missing_flags = {
                f"{col}_was_missing": X_out[col].isna().astype(int)
                for col in self.missing_indicator_columns_
            }

        for column, fill_value in self.fill_values_.items():
            if pd.isna(fill_value):
                raise ValueError(
                    f"No valid fill value was learned for '{column}'. "
                    "The column may have been entirely missing during fit."
                )
            X_out[column] = X_out[column].fillna(fill_value)

        if self.add_missing_indicator:
            for indicator_name, indicator_values in missing_flags.items():
                X_out[indicator_name] = indicator_values.to_numpy()

        return X_out


## 5. Fit on Training Data Only

In [5]:

imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
)

# IMPORTANT: fit() is called only on X_train.
imputer.fit(X_train)

fill_values_table = pd.DataFrame({
    "column": list(imputer.fill_values_.keys()),
    "strategy": [imputer.strategies_[c] for c in imputer.feature_names_in_],
    "fill_value": [imputer.fill_values_[c] for c in imputer.feature_names_in_],
})

display(fill_values_table)

print("Columns that had missing values in training:")
print(imputer.missing_indicator_columns_)


,column,strategy,fill_value
0,student_id,median,1282.0000
1,city,most_frequent,Mumbai
2,city_tier,median,1.0000
3,course,most_frequent,NEET
4,batch_type,most_frequent,Weekday
5,age,median,17.0000
6,enrollment_date,most_frequent,2024-10-23
7,attendance_pct,median,78.3500
8,weekly_study_hours,median,5.0000
9,income_bracket,most_frequent,5-10L


Columns that had missing values in training:
['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']


## 6. Transform Both Training and Test Splits

In [6]:

X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Original training object unchanged:", X_train.isna().sum().sum() > 0)
print("Original test object unchanged:    ", X_test.isna().sum().sum() > 0)

print("\nTransformed training shape:", X_train_imputed.shape)
print("Transformed test shape:    ", X_test_imputed.shape)

print("\nNew indicator columns:")
indicator_columns = [c for c in X_train_imputed.columns if c.endswith("_was_missing")]
print(indicator_columns)


Original training object unchanged: True
Original test object unchanged:     True

Transformed training shape: (480, 22)
Transformed test shape:     (120, 22)

New indicator columns:
['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']


## 7. Verification — No Missing Values Remain

In [7]:

# These are the original columns that required imputation during training.
imputed_columns = imputer.missing_indicator_columns_

train_remaining_missing = X_train_imputed[imputed_columns].isna().sum().sum()
test_remaining_missing = X_test_imputed[imputed_columns].isna().sum().sum()

print("Remaining missing values in imputed training columns:", train_remaining_missing)
print("Remaining missing values in imputed test columns:    ", test_remaining_missing)

assert train_remaining_missing == 0, "Training data still contains missing values."
assert test_remaining_missing == 0, "Test data still contains missing values."

# The complete transformed dataset should also contain no missing values
# because every original column is handled by the custom imputer.
assert X_train_imputed.isna().sum().sum() == 0
assert X_test_imputed.isna().sum().sum() == 0

print("\nPASS: No missing values remain after transformation.")


Remaining missing values in imputed training columns: 0
Remaining missing values in imputed test columns:     0

PASS: No missing values remain after transformation.



## 8. Missing-Value Indicator Verification



In [8]:

indicator_check = pd.DataFrame({
    "indicator": indicator_columns,
    "train_ones": [int(X_train_imputed[c].sum()) for c in indicator_columns],
    "test_ones": [int(X_test_imputed[c].sum()) for c in indicator_columns],
})

display(indicator_check)

for c in indicator_columns:
    assert set(X_train_imputed[c].unique()).issubset({0, 1})
    assert set(X_test_imputed[c].unique()).issubset({0, 1})

print("PASS: All missing indicators are binary.")


,indicator,train_ones,test_ones
0,weekly_study_hours_was_missing,25,11
1,income_bracket_was_missing,24,6
2,prev_exam_score_was_missing,23,2
3,mock_test_3_was_missing,15,6
4,feedback_text_was_missing,54,14


PASS: All missing indicators are binary.


## 9. Numeric Columns — Before vs. After Imputation

In [9]:

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

before_stats = X_train[numeric_columns].agg(["mean", "std"]).T
after_stats = X_train_imputed[numeric_columns].agg(["mean", "std"]).T

numeric_comparison = pd.DataFrame({
    "before_mean": before_stats["mean"],
    "before_std": before_stats["std"],
    "after_mean": after_stats["mean"],
    "after_std": after_stats["std"],
})

numeric_comparison["mean_change"] = (
    numeric_comparison["after_mean"] - numeric_comparison["before_mean"]
)
numeric_comparison["std_change"] = (
    numeric_comparison["after_std"] - numeric_comparison["before_std"]
)

display(numeric_comparison)

print(
    "Comment: columns with no missing values have unchanged statistics. "
    "For columns with missing values, the mean/std can change because missing "
    "observations are replaced by the training-set statistic. Median imputation "
    "typically reduces dispersion because inserted values equal the median."
)


,before_mean,before_std,after_mean,after_std,mean_change,std_change
student_id,1290.5625,173.8254,1290.5625,173.8254,0.0000,0.0000
city_tier,1.4021,0.4908,1.4021,0.4908,0.0000,0.0000
age,16.9563,1.7315,16.9563,1.7315,0.0000,0.0000
attendance_pct,78.5463,13.4596,78.5463,13.4596,0.0000,0.0000
weekly_study_hours,6.0642,4.3503,6.0088,4.2418,-0.0554,-0.1084
prev_exam_score,65.8245,14.3708,65.8377,14.0217,0.0132,-0.3491
mock_test_1,65.9452,16.5662,65.9452,16.5662,0.0000,0.0000
mock_test_2,67.8863,18.1088,67.8863,18.1088,0.0000,0.0000
mock_test_3,70.7015,19.0773,70.7202,18.7765,0.0187,-0.3008
doubt_sessions_attended,4.0042,2.0259,4.0042,2.0259,0.0000,0.0000


Comment: columns with no missing values have unchanged statistics. For columns with missing values, the mean/std can change because missing observations are replaced by the training-set statistic. Median imputation typically reduces dispersion because inserted values equal the median.



## 10. Sanity Check Against scikit-learn `SimpleImputer`



In [10]:

numeric_imputer = SimpleImputer(strategy="median")
numeric_imputer.fit(X_train[numeric_columns])

sklearn_fill_values = pd.Series(
    numeric_imputer.statistics_,
    index=numeric_columns,
    name="sklearn_fill_value"
)

custom_numeric_fill_values = pd.Series(
    {c: imputer.fill_values_[c] for c in numeric_columns},
    name="custom_fill_value"
)

fill_value_comparison = pd.concat(
    [custom_numeric_fill_values, sklearn_fill_values],
    axis=1
)

fill_value_comparison["match"] = np.isclose(
    fill_value_comparison["custom_fill_value"].astype(float),
    fill_value_comparison["sklearn_fill_value"].astype(float),
    equal_nan=True,
)

display(fill_value_comparison)

assert fill_value_comparison["match"].all()
print("PASS: CustomImputer and SimpleImputer produce the same numeric fill values.")


,custom_fill_value,sklearn_fill_value,match
student_id,1282.0000,1282.0000,True
city_tier,1.0000,1.0000,True
age,17.0000,17.0000,True
attendance_pct,78.3500,78.3500,True
weekly_study_hours,5.0000,5.0000,True
prev_exam_score,66.1000,66.1000,True
mock_test_1,66.5500,66.5500,True
mock_test_2,66.3500,66.3500,True
mock_test_3,71.3000,71.3000,True
doubt_sessions_attended,4.0000,4.0000,True


PASS: CustomImputer and SimpleImputer produce the same numeric fill values.



## 11. Reflection Questions

### 1. Why must `fit()` be called only on the training split?

`fit()` learns the statistics used to replace missing values, such as the median or mean. If those statistics are calculated using the full dataset or the test set, information from data that is supposed to simulate unseen future observations leaks into the preprocessing step. That makes evaluation overly optimistic because the model pipeline has indirectly seen information from the test distribution. Therefore, the imputer must be fitted on the training split only, while the same learned values are reused to transform both training and test data.

### 2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem, and what does the indicator add?

No. Mean or median imputation replaces the missing values mechanically, but MNAR means the probability of missingness is related to the unobserved value itself, so the missingness mechanism contains information that the replacement statistic cannot recover. The `mock_test_3_was_missing` indicator preserves a separate signal showing which observations originally had a missing score. This can help a downstream model learn that missingness itself is informative, although an indicator alone does not fully solve the underlying MNAR bias.

### 3. What happens if a brand-new column, entirely missing in training but present in test, is passed to the imputer?

In this implementation, a brand-new test-time column causes `transform()` to raise a clear schema-mismatch error because the input columns must match those seen during `fit()`. If a known training column is entirely missing during `fit()`, its learned fill value is invalid (`NaN`) and `transform()` raises an explicit error rather than silently leaving missing values. A production-grade implementation should define a deliberate schema-drift policy—for example, reject unexpected columns with a clear diagnostic, and use a documented fallback or remove/disable a feature that has no learnable training statistic.



## 12. Optional Bonus — Per-Column Strategy Override

The assignment asks for a `column_overrides` option and a demonstration using at least two columns with different strategies.

Here:
- `weekly_study_hours` uses **mean** instead of the dataset-wide median.
- `income_bracket` explicitly uses **most_frequent**.


In [11]:

bonus_imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
    column_overrides={
        "weekly_study_hours": "mean",
        "income_bracket": "most_frequent",
    },
)

bonus_imputer.fit(X_train)
X_train_bonus = bonus_imputer.transform(X_train)
X_test_bonus = bonus_imputer.transform(X_test)

bonus_demo = pd.DataFrame({
    "column": ["weekly_study_hours", "income_bracket"],
    "strategy_used": [
        bonus_imputer.strategies_["weekly_study_hours"],
        bonus_imputer.strategies_["income_bracket"],
    ],
    "learned_fill_value": [
        bonus_imputer.fill_values_["weekly_study_hours"],
        bonus_imputer.fill_values_["income_bracket"],
    ],
})

display(bonus_demo)

assert bonus_imputer.strategies_["weekly_study_hours"] == "mean"
assert bonus_imputer.strategies_["income_bracket"] == "most_frequent"
assert X_train_bonus.isna().sum().sum() == 0
assert X_test_bonus.isna().sum().sum() == 0

print("PASS: Per-column overrides work with two different strategies.")


,column,strategy_used,learned_fill_value
0,weekly_study_hours,mean,6.0642
1,income_bracket,most_frequent,5-10L


PASS: Per-column overrides work with two different strategies.
